# Cyclistic Bike-Share — Process Phase

Combine the 12 monthly trip files (Jan–Dec 2025), clean known data-quality issues identified
in the Prepare phase, and engineer the columns needed for analysis (`ride_length_min`,
`day_of_week`, `day_name`, `month`, `hour`).

Source data: `data/raw/*.csv` (not committed, see `data/raw/README.md`).
Output: `data/processed/all_trips_2025.parquet` (not committed — large; regenerate by running
this notebook).

## Step 1 — Load and combine the 12 monthly files

In [1]:
import pandas as pd
import glob
import os

RAW_DIR = "../data/raw"
OUT_DIR = "../data/processed"
os.makedirs(OUT_DIR, exist_ok=True)

dtype = {
    "ride_id": "string", "rideable_type": "category",
    "start_station_name": "string", "start_station_id": "string",
    "end_station_name": "string", "end_station_id": "string",
    "member_casual": "category",
}
usecols = ["ride_id", "rideable_type", "started_at", "ended_at",
           "start_station_name", "end_station_name", "member_casual"]

files = sorted(glob.glob(os.path.join(RAW_DIR, "*.csv")))
frames = [pd.read_csv(f, dtype=dtype, usecols=usecols, parse_dates=["started_at", "ended_at"]) for f in files]
trips = pd.concat(frames, ignore_index=True)
print(f"Loaded {len(files)} files -> combined shape: {trips.shape}")

Loaded 12 files -> combined shape: (5552994, 7)

## Step 2 — Remove rows with a negative ride duration

The Prepare-phase integrity check found 29 rows where `ended_at` is earlier than `started_at`
— a data-entry / clock anomaly, not a real ride.

In [2]:
before = len(trips)
trips = trips[trips["ended_at"] >= trips["started_at"]].copy()
after = len(trips)
print(f"Dropped {before - after} rows (ended_at < started_at). Remaining: {after}")

Dropped 29 rows (ended_at < started_at). Remaining: 5552965

## Step 3 — Feature engineering

- `ride_length_min`: trip duration in minutes.
- `day_of_week`: numeric, matching the case study's `WEEKDAY(date, 1)` convention
  (1 = Sunday ... 7 = Saturday).
- `day_name`, `month`, `hour`: convenience columns for the Analyze phase.

In [3]:
trips["ride_length_min"] = (trips["ended_at"] - trips["started_at"]).dt.total_seconds() / 60
trips["day_of_week"] = trips["started_at"].dt.dayofweek.map(lambda x: (x + 1) % 7 + 1)
trips["day_name"] = trips["started_at"].dt.day_name()
trips["month"] = trips["started_at"].dt.month
trips["hour"] = trips["started_at"].dt.hour

trips[["started_at", "ride_length_min", "day_of_week", "day_name", "month", "hour"]].head(3)

               started_at  ride_length_min  day_of_week  day_name  month  hour
0 2025-01-21 17:23:54.538        13.957950            3   Tuesday      1    17
1 2025-01-11 15:44:06.795         5.072400            7  Saturday      1    15
2 2025-01-02 15:16:27.730        11.591667            5  Thursday      1    15

## Step 4 — Check the `ride_length_min` distribution for outliers

In [4]:
print(trips["ride_length_min"].describe(percentiles=[.01, .25, .5, .75, .95, .99, .999]))

extreme_long = (trips["ride_length_min"] > 1440).sum()  # over 24h
very_short = (trips["ride_length_min"] < 1).sum()        # under 1 minute
print(f"\nRides longer than 24h: {extreme_long} ({extreme_long/after*100:.3f}%)")
print(f"Rides shorter than 1 min: {very_short} ({very_short/after*100:.3f}%)")

count    5.552965e+06
mean     1.602802e+01
std      5.511648e+01
min      7.666667e-04
1%       2.597000e-01
25%      5.395000e+00
50%      9.426100e+00
75%      1.656328e+01
95%      3.970556e+01
99%      8.903636e+01
99.9%    1.457677e+03
max      1.574900e+03
Name: ride_length_min, dtype: float64

Rides longer than 24h: 5585 (0.101%)
Rides shorter than 1 min: 147372 (2.654%)

## Step 4b — Remove outlier rides

**Decision:** drop rides under 1 minute and over 24 hours (2.75% of rows combined).

- Rides **under 1 minute** are very likely false starts or a bike being redocked immediately
  (e.g. staff rebalancing, or a rider changing their mind) — not a genuine trip.
- Rides **over 24 hours** very likely indicate a bike that was not properly returned, rather than
  an actual multi-day rental.

Both would distort ride-length averages without representing typical member/casual usage, which
is exactly the metric this analysis needs to compare.

In [5]:
before_outliers = len(trips)
trips = trips[(trips["ride_length_min"] >= 1) & (trips["ride_length_min"] <= 1440)].copy()
after_outliers = len(trips)
print(f"Removed {before_outliers - after_outliers} outlier rides "
      f"({(before_outliers - after_outliers) / before_outliers * 100:.2f}%). "
      f"Remaining: {after_outliers}")

Removed 152957 outlier rides (2.75%). Remaining: 5400008

## Step 5 — Check the member vs. casual split

In [6]:
print(trips["member_casual"].value_counts())
print(trips["member_casual"].value_counts(normalize=True) * 100)

member_casual
member    3484202
casual    1915806
Name: count, dtype: int64
member_casual
member    64.522164
casual    35.477836
Name: proportion, dtype: float64

## Step 6 — Save the cleaned, merged dataset

Saved as Parquet (columnar, compressed) rather than CSV — keeps the ~5.4M-row file manageable
and fast to reload for the Analyze phase. Excluded from version control (see repo `.gitignore`);
regenerate by re-running this notebook against `data/raw/`.

In [7]:
out_path = os.path.join(OUT_DIR, "all_trips_2025.parquet")
trips.to_parquet(out_path, index=False)
size_mb = os.path.getsize(out_path) / 1e6
print(f"Saved {len(trips)} rows to {out_path} ({size_mb:.1f} MB)")

Saved 5400008 rows to ../data/processed/all_trips_2025.parquet (236.9 MB)

## Summary

| Step | Rows removed | Reason |
|---|---|---|
| Negative duration | 29 | `ended_at` before `started_at` (data anomaly) |
| Duration < 1 min | (combined below) | Likely false start / immediate redock |
| Duration > 24h | (combined below) | Likely bike not properly returned |
| **Total removed** | **152,986** (2.76%) | |
| **Final dataset** | **5,400,008 rows** | Ready for the Analyze phase |

Final split: **64.5% members / 35.5% casual riders** across the cleaned 2025 dataset.